# NB03: Cultured Genome KO Profiles

Extract per-genome KEGG Ortholog profiles for all 3,110 ENIGMA Genome Depot genomes.

**Requires**: BERDL Spark access (JupyterHub or on-cluster)

In [ ]:
import os, pandas as pd, numpy as np
from berdl_notebook_utils.setup_spark_session import get_spark_session

spark = get_spark_session()
DATA_DIR = '../data'
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Genome Metadata

In [ ]:
genome_meta = spark.sql('''
    SELECT 
        bg.id as genome_id, bg.name as genome_name,
        bg.contigs, bg.size as genome_size, bg.genes as n_genes,
        bg.external_id,
        bt.name as taxon_name, bt.rank as taxon_rank, bt.taxonomy_id
    FROM enigma_genome_depot_enigma.browser_genome bg
    LEFT JOIN enigma_genome_depot_enigma.browser_taxon bt ON bg.taxon_id = bt.id
''')

gm_pd = genome_meta.toPandas()
gm_pd.to_csv(f'{DATA_DIR}/cultured_genome_metadata.tsv', sep='\t', index=False)
print(f'{len(gm_pd)} genomes')
gm_pd.head()

## 2. Genome-KO Profiles

In [ ]:
genome_ko = spark.sql('''
    SELECT 
        bge.genome_id,
        bko.kegg_id as ko_id,
        bko.description as ko_description,
        COUNT(*) as n_copies
    FROM enigma_genome_depot_enigma.browser_gene bge
    JOIN enigma_genome_depot_enigma.browser_protein bp ON bge.protein_id = bp.id
    JOIN enigma_genome_depot_enigma.browser_protein_kegg_orthologs bpko ON bp.id = bpko.protein_id
    JOIN enigma_genome_depot_enigma.browser_kegg_ortholog bko ON bpko.kegg_ortholog_id = bko.id
    GROUP BY bge.genome_id, bko.kegg_id, bko.description
''')

gko_pd = genome_ko.toPandas()
gko_pd.to_csv(f'{DATA_DIR}/cultured_ko_profiles.tsv', sep='\t', index=False)
print(f'{len(gko_pd)} genome-KO pairs')
print(f'{gko_pd["genome_id"].nunique()} genomes, {gko_pd["ko_id"].nunique()} unique KOs')

## 3. KO Summary Statistics

In [ ]:
kos_per_genome = gko_pd.groupby('genome_id')['ko_id'].nunique()
print(f'KOs per genome:')
print(f'  mean={kos_per_genome.mean():.0f}, median={kos_per_genome.median():.0f}')
print(f'  min={kos_per_genome.min()}, max={kos_per_genome.max()}')
print(f'  Q25={kos_per_genome.quantile(0.25):.0f}, Q75={kos_per_genome.quantile(0.75):.0f}')

## 4. Marker Dictionary Coverage

In [ ]:
markers = {
    'Wood-Ljungdahl': ['K00198', 'K00194', 'K00197', 'K15022', 'K15023'],
    'NiFe-hydrogenase': ['K06281', 'K06282'],
    'Sulfate reduction': ['K11180', 'K11181', 'K00394', 'K00395', 'K00958'],
    'Denitrification': ['K00370', 'K00371', 'K02567', 'K00368', 'K15864', 'K00376'],
    'Methanogenesis': ['K00399', 'K00401', 'K00402'],
    'N-fixation': ['K02588', 'K02586', 'K02591'],
    'Metal resistance': ['K07787', 'K07665', 'K19592', 'K00537'],
    'Motility/chemotaxis': ['K02406', 'K03406', 'K03407'],
    'Aerobic respiration': ['K02274', 'K02275', 'K02276'],
    'Conjugation/MGE': ['K03204', 'K03197'],
}

n_genomes = gko_pd['genome_id'].nunique()
results = []
for category, kos in markers.items():
    present = gko_pd[gko_pd['ko_id'].isin(kos)]['ko_id'].unique()
    genomes_with = gko_pd[gko_pd['ko_id'].isin(kos)]['genome_id'].nunique()
    pct = genomes_with / n_genomes * 100
    print(f'{category:<25} {len(present)}/{len(kos)} KOs, {genomes_with}/{n_genomes} genomes ({pct:.1f}%)')
    for ko in kos:
        ko_n = gko_pd[gko_pd['ko_id'] == ko]['genome_id'].nunique()
        results.append({'category': category, 'ko_id': ko, 'n_genomes': ko_n, 
                       'pct_genomes': round(ko_n / n_genomes * 100, 2)})

marker_df = pd.DataFrame(results)
marker_df.to_csv(f'{DATA_DIR}/cultured_marker_coverage.tsv', sep='\t', index=False)

## 5. Taxonomic Breakdown

In [ ]:
taxon_counts = gm_pd['taxon_name'].value_counts().head(20)
print('Top 20 taxa in cultured collection:')
for taxon, n in taxon_counts.items():
    print(f'  {taxon}: {n}')